<a href="https://colab.research.google.com/github/Knguyen351/CIS-3400-Advanced-Programming-Kevin-Nguyen/blob/main/Charlotte_Public_Safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Investigating Public Safety Trends in Charlotte
**Name:** Kevin Nguyen  
**Date:** October 16, 2026

## Data description
The filtered dataframe contains **2,597 rows and 27 columns**. The report-date range is **2026-01-01 05:00:00+00:00 to 2026-10-15 04:00:00+00:00**. Incident-began dates range from **2023-04-06 04:00:00+00:00 to 2026-10-15 04:00:00+00:00**. Features include report ID, year, public location and coordinates, patrol division, neighborhood area, report and incident dates, place descriptions, highest NIBRS crime code and description, clearance status, detailed status, clearance date, and technical identifiers. GIS shapes are omitted. The full column list, data types, and nonmissing counts appear in the executed `df.info(show_counts=True, verbose=True)` output below. These numbers describe the saved run; rerun outputs provide updated totals. Download tested at **2026-10-17T02:47:34.275410+00:00**.

## Two questions of interest
1. When comparing Central and Uni City, which patrol division is more involved with reports of Burglary, theft on moter vehicles and simple assault in 2026.

2. When it comes to buglary, theft and simple assaults, how does clearances statueses differ?

## Scope and source
I selected **220 — Burglary/B&E**, **23F — Theft From Motor Vehicle**, and **240 — Motor Vehicle Theft**. The geographic filter includes **Central** and **University City**. The timeline uses **YEAR = '2026'**, the portal's Year field. Actual report and incident date ranges appear below; the Year field should not be assumed to equal every date field's calendar year.

Source: [City of Charlotte CMPD Incidents portal](https://data.charlottenc.gov/datasets/charlotte::cmpd-incidents-1/explore). [Official query service and field definitions](https://gis.charlottenc.gov/arcgis/rest/services/CMPD/CMPDIncidents/MapServer/0).

This is a live, incomplete 2026 snapshot. Rerunning can change the counts and clearance statuses. Each report is classified by its highest NIBRS offense, so these counts do not include every offense within every report. The source also includes unfounded reports; I retain them and display their statuses. Counts are not population-adjusted crime rates. A clearance status is not evidence of a conviction, and recent cases have had less time to be cleared.

## 1. Setup
Run the cells from top to bottom. Internet access and pandas are required. If pandas is missing, run `%pip install pandas` in a separate cell, then restart the kernel.

`json` reads the API response, `urllib` sends the web request, and `pandas` organizes the results into a dataframe.

In [1]:
import json
import time
from urllib.parse import urlencode
from urllib.request import urlopen
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

LAYER_URL = "https://gis.charlottenc.gov/arcgis/rest/services/CMPD/CMPDIncidents/MapServer/0"
YEAR = 2026
CRIME_CODES = ["13B", "23F", "138"]
DIVISIONS = ["Central", "University City"]

def get_json(url, params):
    # Retry temporary connection problems; never silently substitute old data.
    for attempt in range(3):
        try:
            with urlopen(url + "?" + urlencode(params), timeout=60) as response:
                result = json.load(response)
            break
        except (OSError, TimeoutError, json.JSONDecodeError):
            if attempt == 2:
                raise
            time.sleep(2)
    if "error" in result:
        raise RuntimeError(result["error"])
    return result

def sql_text(value):
    # Quote text safely for the ArcGIS SQL filter.
    return "'" + str(value).replace("'", "''") + "'"

where = (
    f"YEAR = {sql_text(YEAR)} AND "
    f"HIGHEST_NIBRS_CODE IN ({','.join(map(sql_text, CRIME_CODES))}) AND "
    f"CMPD_PATROL_DIVISION IN ({','.join(map(sql_text, DIVISIONS))})"
)
print("Filter:", where)

Filter: YEAR = '2026' AND HIGHEST_NIBRS_CODE IN ('13B','23F','138') AND CMPD_PATROL_DIVISION IN ('Central','University City')


## 2. Download all matching records into `df`
The service limits the number of rows in one response. This code first requests all matching object IDs, then downloads them in batches. It checks that every requested ID was returned. `returnGeometry=false` skips GIS shapes; public coordinate columns are retained.

In [2]:
metadata = get_json(LAYER_URL, {"f": "json"})
query_url = LAYER_URL + "/query"
id_result = get_json(query_url, {"f": "json", "where": where, "returnIdsOnly": "true"})
object_ids = sorted(id_result.get("objectIds") or [])
if not object_ids:
    raise ValueError("No matching records. Check the selected year, codes, and divisions.")
id_field = id_result["objectIdFieldName"]
rows = []
batch_size = min(100, metadata.get("maxRecordCount", 100))
for start in range(0, len(object_ids), batch_size):
    batch = object_ids[start:start + batch_size]
    result = get_json(query_url, {
        "f": "json", "objectIds": ",".join(map(str, batch)),
        "outFields": "*", "returnGeometry": "false"
    })
    if result.get("exceededTransferLimit"):
        raise RuntimeError("Server truncated a batch; reduce batch_size.")
    rows.extend(feature["attributes"] for feature in result["features"])

df = pd.DataFrame(rows)
assert len(df) == len(object_ids), "Downloaded row count does not match requested IDs."
assert df[id_field].is_unique, "Duplicate object IDs were returned."
assert set(df[id_field]) == set(object_ids), "Some requested records were not returned."
assert df["YEAR"].eq(str(YEAR)).all()
assert df["HIGHEST_NIBRS_CODE"].isin(CRIME_CODES).all()
assert df["CMPD_PATROL_DIVISION"].isin(DIVISIONS).all()

# ArcGIS dates are Unix milliseconds. Keep UTC to avoid silently shifting dates.
date_fields = [f["name"] for f in metadata["fields"] if f["type"] == "esriFieldTypeDate"]
for column in date_fields:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], unit="ms", utc=True, errors="coerce")
df = df.sort_values(id_field).reset_index(drop=True)
retrieved_at = datetime.now(timezone.utc).isoformat()
print(f"Retrieved {len(df):,} rows and {df.shape[1]} columns at {retrieved_at}.")
display(df.head())

Retrieved 2,699 rows and 27 columns at 2026-10-17T03:39:07.943448+00:00.


,YEAR,INCIDENT_REPORT_ID,LOCATION,CITY,STATE,ZIP,X_COORD_PUBLIC,Y_COORD_PUBLIC,LATITUDE_PUBLIC,LONGITUDE_PUBLIC,...,LOCATION_TYPE_DESCRIPTION,PLACE_TYPE_DESCRIPTION,PLACE_DETAIL_DESCRIPTION,CLEARANCE_STATUS,CLEARANCE_DETAIL_STATUS,CLEARANCE_DATE,HIGHEST_NIBRS_CODE,HIGHEST_NIBRS_DESCRIPTION,OBJECTID,GlobalID
0,2026,20260731-1809-02,8500 UNIVERSITY EXECUTIVE PARK DR,CHARLOTTE,NC,28262,1476459,569108,35.301014,-80.754806,...,Parking Lot,Residential,Apartment/Duplex Private Res,Open,Open,NaT,23F,Theft From Motor Vehicle,175,{A2B515AB-AE14-4DAE-8476-F613FB5C757A}
1,2026,20260607-1424-01,8000 SIR HENRY ST,CHARLOTTE,NC,28262,1477624,579587,35.329859,-80.751522,...,Indoors,Residential,Apartment/Duplex Private Res,Open,Open,NaT,13B,Simple Assault,417,{DEBF52A9-AE52-4BC8-802D-47741370F25E}
2,2026,20260901-0839-03,500 W 10TH ST,CHARLOTTE,NC,None,1450454,545641,35.235252,-80.840492,...,Parking Lot,Residential,Apartment/Duplex Private Res,Open,Open,NaT,23F,Theft From Motor Vehicle,1019,{78778F9D-3E95-4BB1-B5F0-B970B5FCFE96}
3,2026,20260214-2336-00,100 E 6TH ST,CHARLOTTE,NC,28202,1450307,543285,35.228772,-80.840838,...,Outdoors,Open Area,Street/Highway,Open,Open,NaT,13B,Simple Assault,1264,{EA258ECD-E73C-44C0-B2EC-519B004A45A9}
4,2026,20260505-2039-00,800 AVETT LN,CHARLOTTE,NC,28262,1492571,586439,35.349397,-80.701809,...,Parking Lot,Residential,Apartment/Duplex Private Res,Open,Open,NaT,23F,Theft From Motor Vehicle,2123,{694D2442-31BB-4AED-BE96-2D736605D61A}


## 3. Inspect the data
`df.info()` reports column names, nonmissing counts, data types, and memory use. The date table reports actual minimum and maximum dates rather than assuming a full year of coverage.

In [3]:
df.info(show_counts=True, verbose=True)

date_summary = pd.DataFrame({
    "Earliest (UTC)": df[date_fields].min(),
    "Latest (UTC)": df[date_fields].max(),
    "Missing values": df[date_fields].isna().sum()
})
display(date_summary)
print("Unique incident report IDs:", df["INCIDENT_REPORT_ID"].nunique())
print("Duplicate incident report IDs:", df["INCIDENT_REPORT_ID"].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2699 entries, 0 to 2698
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   YEAR                       2699 non-null   object             
 1   INCIDENT_REPORT_ID         2699 non-null   object             
 2   LOCATION                   2699 non-null   object             
 3   CITY                       2699 non-null   object             
 4   STATE                      2699 non-null   object             
 5   ZIP                        1493 non-null   object             
 6   X_COORD_PUBLIC             2699 non-null   int64              
 7   Y_COORD_PUBLIC             2699 non-null   int64              
 8   LATITUDE_PUBLIC            2699 non-null   float64            
 9   LONGITUDE_PUBLIC           2699 non-null   float64            
 10  DIVISION_ID                2699 non-null   object             
 11  CMPD

,Earliest (UTC),Latest (UTC),Missing values
DATE_REPORTED,2026-01-01 05:00:00+00:00,2026-10-15 04:00:00+00:00,0
DATE_INCIDENT_BEGAN,2021-01-29 05:00:00+00:00,2026-10-15 04:00:00+00:00,0
DATE_INCIDENT_END,2023-11-12 05:00:00+00:00,2026-10-15 04:00:00+00:00,309
CLEARANCE_DATE,2026-01-01 05:00:00+00:00,2026-10-15 04:00:00+00:00,1956


Unique incident report IDs: 2699
Duplicate incident report IDs: 0


## 4. Explore the two questions
The first table compares report counts by division and crime. The next tables show clearance categories and their percentages within each crime type. Missing statuses are labeled explicitly. These are descriptive snapshots, not a measure of police effectiveness.

In [4]:
crime_counts = pd.crosstab(df["CMPD_PATROL_DIVISION"], df["HIGHEST_NIBRS_DESCRIPTION"])
display(crime_counts)
clearance_counts = pd.crosstab(
    df["HIGHEST_NIBRS_DESCRIPTION"], df["CLEARANCE_STATUS"].fillna("Missing")
)
display(clearance_counts)
display(clearance_counts.div(clearance_counts.sum(axis=1), axis=0).mul(100).round(1))
display(pd.crosstab(
    df["HIGHEST_NIBRS_DESCRIPTION"], df["CLEARANCE_DETAIL_STATUS"].fillna("Missing")
))

HIGHEST_NIBRS_DESCRIPTION,Simple Assault,Theft From Motor Vehicle
CMPD_PATROL_DIVISION,,
Central,473,546
University City,872,808


CLEARANCE_STATUS,Cleared by Arrest,Cleared by Arrest by Another Agency,Exceptionally Cleared,Open,Unfounded
HIGHEST_NIBRS_DESCRIPTION,,,,,
Simple Assault,314,27,192,793,19
Theft From Motor Vehicle,43,7,4,1256,44


CLEARANCE_STATUS,Cleared by Arrest,Cleared by Arrest by Another Agency,Exceptionally Cleared,Open,Unfounded
HIGHEST_NIBRS_DESCRIPTION,,,,,
Simple Assault,23.3,2.0,14.3,59.0,1.4
Theft From Motor Vehicle,3.2,0.5,0.3,92.8,3.2


CLEARANCE_DETAIL_STATUS,Cleared by Arrest,Cleared by Arrest by Another Agency,Cleared by Other Means,"Cleared, Pending Arrest Validation",Open,Prosecution Declined by DA,Unfounded,Unfounded-Referred to Other Agency,Victim Chose not to Prosecute
HIGHEST_NIBRS_DESCRIPTION,,,,,,,,,
Simple Assault,314,27,31,74,719,2,15,4,159
Theft From Motor Vehicle,43,7,1,19,1237,0,44,0,3


## 5. Export the filtered data
This creates a CSV in the same working folder as the notebook. It includes the selected records, not the entire city dataset.

In [5]:
csv_name = f"CMPD_property_crimes_{YEAR}.csv"
df.to_csv(csv_name, index=False)
print("Saved:", csv_name)

Saved: CMPD_property_crimes_2026.csv
